In [1]:
APPLICATION_NAME = "surveillance_certfr"

In [ ]:
import sys
from pathlib import Path

# Remonter les dossiers jusqu'à trouver lib/init/init_code.py
def find_app_home(sentinel: str ="lib/bootstrap/bootstrap.py"):
    current = Path.cwd().resolve()
    root = current.root
    while current != root:
        if (current / sentinel).is_file():
            return current
        current = current.parent
    raise FileNotFoundError(f"Impossible de trouver le fichier sentinelle : {sentinel}")

# Trouver et ajouter APPLICATION_HOME au sys.path
APPLICATION_HOME = find_app_home()
sys.path.insert(0, str(APPLICATION_HOME))
#print(f"APPLICATION_HOME set to: {APPLICATION_HOME}")

from lib.bootstrap.bootstrap import init_env
epy = init_env()

In [17]:
# import
import feedparser
import smtplib
from email.mime.multipart import MIMEMultipart      # MIMEMultipart send emails with both text content and attachments.
from email.mime.text import MIMEText                # MIMEText for creating body of the email message.
from email.mime.application import MIMEApplication  # MIMEApplication attaching application-specific data (like CSV files) to email messages.
from email.utils import COMMASPACE

In [ ]:
# variables
log = epy.log
yml = epy.cfgyaml

watch = yml.get("watch_list")
# print(watch)
excluding = yml.get("exclude")
# print(excluding)

## Traitement

In [ ]:
log.log("==========================================")
log.log("=== Scan des avis & alertes du CERT-FR ===")
log.log("==========================================")
log.info(f"Liste des programmes surveillés : {watch}")
log.log("")

### Recherche des alertes et avis CERTFR

In [ ]:
for feed in yml.get("feed"):

    log.info(f"{feed['name'].capitalize()} de sécurité: {feed['url']} ")
    parse = feedparser.parse(f"{feed['url']}/feed/")

    for p in parse.entries:
        # print(f"{p['title']} - {p['link']}")
        id = p['link'].strip(feed["url"])
        title_lowercase = p['title'].lower()
        if any(terme in title_lowercase for terme in watch):
            log.log(f"{p['title']} - {id} -  {p['link']}")
    log.log("")
    

In [ ]:
log.log("=== Fin du traitement ===")
log.log("")

### Alerting par email

In [ ]:
subject = yml.get('email.sujet')
# print(subject)
body = "This is the body of the text message"
sender_email = yml.get('smtp.user')
# print(sender_email)
recipient_email = COMMASPACE.join(yml.get('email.to'))
# print(recipient_email)
sender_password = yml.get('smtp.password')
# print(sender_password)
smtp_server = yml.get('smtp.host')
# print(smtp_server)
smtp_port = yml.get('smtp.port')
# print(smtp_port)
path_to_file = log.get_log_file
# print(path_to_file)

# MIMEMultipart() creates a container for an email message that can hold
# different parts, like text and attachments and in next line we are
# attaching different parts to email container like subject and others.
message = MIMEMultipart()
message['Subject'] = subject
message['From'] = sender_email
message['To'] = recipient_email
body_part = MIMEText(body)
message.attach(body_part)


print(message)

# # section 1 to attach file
# with open(path_to_file,'rb') as file:
#     # Attach the file with filename to the email
#     message.attach(MIMEApplication(file.read(), Name="example.csv"))

# secction 2 for sending email
with smtplib.SMTP(smtp_server, smtp_port) as server:
   if yml.get('isTls'):
      server.starttls()
   server.login(sender_email, sender_password)
   server.sendmail(sender_email, recipient_email, message.as_string())